In [252]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier


In [253]:
def label_from_filename(filename: int):
    
    ranges = {
        (1001, 1059): 1,
        (1060, 1122): 2,
        (1552, 1616): 3,
        (1123, 1194): 4,
        (1195, 1267): 5,
        (1268, 1323): 6,
        (1324, 1385): 7,
        (1386, 1437): 8,
        (1438, 1496): 10,
        (1497, 1551): 9,
        (2001, 2050): 11,
        (2051, 2113): 12,
        (2114, 2165): 14,
        (2166, 2230): 15,
        (2231, 2290): 16,
        (2291, 2346): 17,
        (2347, 2423): 18,
        (2424, 2485): 19,
        (2486, 2546): 20,
        (2547, 2612): 21,
        (2616, 2675): 22,
        (3001, 3055): 23,
        (3056, 3110): 24,
        (3111, 3175): 25,
        (3176, 3229): 26,
        (3230, 3281): 27,
        (3282, 3334): 28,
        (3335, 3389): 29,
        (3390, 3446): 30,
        (3447, 3510): 31,
        (3511, 3563): 32,
        (3566, 3621): 33,
    }

    for (low, high), label in ranges.items():
        if low <= filename <= high:
            return label
    
    return None


In [254]:
def process_image(image:str):

    image_filename = os.path.basename(image)

    #Loading image
    im = cv.cvtColor(cv.imread(image), cv.COLOR_BGR2GRAY)

    # Change size

    im = cv.resize(im, (1000, 1000))
    
    #Image segmentation using otsu
    im_segmentation = cv.threshold(im, 0,255, cv.THRESH_BINARY+cv.THRESH_OTSU)
    
    #Contour extraction
    contours, hierarchy = cv.findContours(im_segmentation[1], cv.RETR_LIST, cv.CHAIN_APPROX_NONE)

    max_area = -1
    max_contour = None
    image_area = im.shape[0] * im.shape[1]
    for contour in contours:
    
        area = cv.contourArea(contour)

        # Drawing a bounding box
        if 2000 < area <= image_area * 0.9:
            if  area > max_area:
                max_area = area
                max_contour = contour
            

    x, y, w, h = cv.boundingRect(max_contour)


    # Perimeter calculation

    perim = cv.arcLength(max_contour, True) # Verificar

    # Aspect ratio calculation
    aspect_ration = w/h

    # Shape factor
    shape_factor = (4*np.pi*max_area)/np.power(perim, 2)

    #
    rectan = (w*h)/max_area

    points = np.array(max_contour).reshape(-1,2).astype(np.float32)
    mean, eigvec, eigval = cv.PCACompute2(points, mean=None)    


    eje_mayor = float(2 * np.sqrt(eigval[0][0])) # Eje mayor
    eje_menor = float(2 * np.sqrt(eigval[1][0])) # Eje menor

    # Perimeter diameter ratio

    p_d_ratio = perim/w

    # Hu moments

    hu = cv.HuMoments(cv.moments(max_contour)).flatten()

    # Label for image

    filename_id = int(image_filename[:4])

    classification = label_from_filename(filename_id)

    # Our vector will consists of: [A, P, SF, R, w, h, aspect_ratio, eje_mayor, eje_menor, p_d_ratio, hu]

    features = np.hstack([
            max_area, perim, shape_factor, rectan,
        w, h, aspect_ration, eje_mayor, eje_menor, p_d_ratio, hu
       ])

    return features, classification

    



In [255]:
def obtain_features_from_dataset(path:str):

    X = []
    y = []
    # Verify path exists.


    if not os.path.exists(path):
        print("Error")
        return
    
    # Iterate over files and process them.
    # We need files sorted 

    for x in os.listdir(path):
        image_path = os.path.join(path, x)
        features, classification = process_image(image_path)
        X.append(features)
        y.append(classification)


    
    return np.array(X), np.array(y)
        
    


## Exctraction feature vectors

In [256]:
path = "./Leaves"
X, y = obtain_features_from_dataset(path)

In [257]:
# Scaling dataset

scaler = StandardScaler() 

X = scaler.fit_transform(X)



# Train test split

In [258]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

# Testing different models

Now we are going to try different models like SVM, KNN and MLP on the dataset using the feature vector we build in past sections.

## 1) SVM
### Searching best model

In [259]:
params = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.1, 0.01],
    "kernel": ["rbf"]
}

grid = GridSearchCV(SVC(), params, cv=5)
grid.fit(X_train, y_train)

print(grid.best_params_)


{'C': 100, 'gamma': 0.1, 'kernel': 'rbf'}


In [260]:
svm_classifier = SVC(kernel="rbf", C=100, gamma="scale")

svm_classifier.fit(X_train, y_train)

,C,100
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [261]:
y_pred = svm_classifier.predict(X_test)

In [262]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.81


## 2) KNN

In [263]:
params_knn = {
    "n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"],
    "p": [1, 2]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    params_knn,
    cv=5,
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print(grid_knn.best_params_)

{'metric': 'manhattan', 'n_neighbors': 9, 'p': 1, 'weights': 'distance'}


In [264]:
neigh = KNeighborsClassifier(**grid_knn.best_params_)
neigh.fit(X_train, y_train)
y_pred_knn = neigh.predict(X_test)


accuracy_knn = accuracy_score(y_test, y_pred_knn)


print(f"Model accuracy: {accuracy_knn:.2f}")

Model accuracy: 0.72


# 3) MLP

In [265]:
params_mlp = {
    "hidden_layer_sizes": [
        (32,), (64,), (128,),
        (64, 32), (128, 64)
    ],
    "activation": ["relu", "tanh"],
    "solver": ["adam"],
    "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
    "learning_rate": ["constant", "adaptive"],
    "max_iter": [500]
}

grid_mlp = GridSearchCV(
    MLPClassifier(random_state=42),
    params_mlp,
    cv=5,
    n_jobs=-1
)

grid_mlp.fit(X_train, y_train)

print(grid_mlp.best_params_)
print(grid_mlp.best_score_)

/Users/rafael/Documents/image-analysis/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rafael/Documents/image-analysis/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rafael/Documents/image-analysis/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rafael/Documents/image-analysis/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the opti

{'activation': 'tanh', 'alpha': 1e-05, 'hidden_layer_sizes': (128, 64), 'learning_rate': 'constant', 'max_iter': 500, 'solver': 'adam'}
0.7685245901639345


/Users/rafael/Documents/image-analysis/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [266]:
mlp = MLPClassifier(**grid_mlp.best_params_).fit(X_train, y_train)

/Users/rafael/Documents/image-analysis/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [267]:
y_pred_mlp = neigh.predict(X_test)


accuracy_mlp= accuracy_score(y_test, y_pred_mlp)


print(f"Model accuracy: {accuracy_mlp:.2f}")

Model accuracy: 0.72
